# Gộp train và test vào `vsl400-keypoint`

Gắn hai dataset vào Input:

- `vsl400-train-keypoints` (folder `train`)
- `vsl400-test-keypoints` (folder `test`)

Notebook copy cây thư mục sang:

- `/kaggle/working/vsl400-keypoint/train`
- `/kaggle/working/vsl400-keypoint/test`

Ưu tiên hardlink. Cùng ổ thì không tốn thêm dung lượng Output. Khác ổ thì copy file. Không cần Internet, không cần GPU.


In [ ]:
import logging
import os
import shutil
from multiprocessing import get_context
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    force=True,
)
logger = logging.getLogger("pack_vsl400")

INPUT_DIR = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/vsl400-keypoint")
SPLITS = ("train", "test")
SKIP_DIR_NAMES = {"mediapipe_models"}


def find_split_dirs(input_dir: Path) -> dict[str, Path]:
    """Return one source directory for train and one for test.

    A split directory is a folder named train or test. The walk stops there
    so gloss folders are not scanned twice. When several candidates exist,
    the parent whose name contains the split name wins.
    """
    found: dict[str, list[Path]] = {split: [] for split in SPLITS}
    if not input_dir.is_dir():
        logger.error("Input directory does not exist: %s", input_dir)
        return {}

    for dirpath, dirnames, _filenames in os.walk(input_dir):
        dirnames[:] = [
            name
            for name in dirnames
            if name not in SKIP_DIR_NAMES and not name.startswith(".")
        ]
        base = os.path.basename(dirpath)
        if base not in SPLITS:
            continue
        found[base].append(Path(dirpath))
        dirnames.clear()

    chosen: dict[str, Path] = {}
    for split, candidates in found.items():
        if not candidates:
            continue
        candidates.sort(
            key=lambda path: (
                0 if split in path.parent.name.lower() else 1,
                str(path),
            )
        )
        chosen[split] = candidates[0]
        for extra in candidates[1:]:
            logger.warning("Ignoring extra %s source: %s", split, extra)
    return chosen


def collect_jobs(split: str, source: Path) -> list[tuple[str, str]]:
    jobs = []
    dest_root = OUTPUT_ROOT / split
    for dirpath, dirnames, filenames in os.walk(source):
        dirnames[:] = [name for name in dirnames if not name.startswith(".")]
        relative = os.path.relpath(dirpath, source)
        dest_dir = dest_root if relative == "." else dest_root / relative
        for name in filenames:
            if name.startswith("."):
                continue
            jobs.append((os.path.join(dirpath, name), str(dest_dir / name)))
    return jobs


if INPUT_DIR.is_dir():
    logger.info("Mounted inputs: %s", sorted(path.name for path in INPUT_DIR.iterdir()))
else:
    raise RuntimeError(f"Nothing is mounted at {INPUT_DIR}. Attach both datasets.")

sources = find_split_dirs(INPUT_DIR)
for split in SPLITS:
    source = sources.get(split)
    if source is None:
        raise RuntimeError(
            f"No {split}/ folder under {INPUT_DIR}. "
            "Attach vsl400-train-keypoints and vsl400-test-keypoints."
        )
    logger.info("%s source: %s", split, source)

if OUTPUT_ROOT.exists():
    logger.info("Removing previous output: %s", OUTPUT_ROOT)
    shutil.rmtree(OUTPUT_ROOT)

jobs: list[tuple[str, str]] = []
for split in SPLITS:
    split_jobs = collect_jobs(split, sources[split])
    logger.info("%s: %d files", split, len(split_jobs))
    jobs.extend(split_jobs)

if not jobs:
    raise RuntimeError("Both split folders are empty.")

dest_dirs = {os.path.dirname(dst) for _src, dst in jobs}
for directory in sorted(dest_dirs):
    os.makedirs(directory, exist_ok=True)

logger.info("Planned %d files into %s", len(jobs), OUTPUT_ROOT)


In [ ]:
def _link_or_copy(job: tuple[str, str]) -> str:
    src, dst = job
    try:
        os.link(src, dst)
        return "link"
    except OSError:
        shutil.copy2(src, dst)
        return "copy"


worker_count = min(len(jobs), max(1, (os.cpu_count() or 1) - 1))
logger.info("Copying with %d workers", worker_count)
with get_context("fork").Pool(processes=worker_count) as pool:
    results = pool.map(_link_or_copy, jobs, chunksize=64)

link_count = sum(kind == "link" for kind in results)
copy_count = sum(kind == "copy" for kind in results)
logger.info(
    "Placed %d hardlinks and %d copies into %s",
    link_count,
    copy_count,
    OUTPUT_ROOT,
)


In [ ]:
import numpy as np

placed = 0
npy_counts = {split: 0 for split in SPLITS}
gloss_counts = {split: 0 for split in SPLITS}
bad_files = []
checked = 0

for split in SPLITS:
    split_dir = OUTPUT_ROOT / split
    for dirpath, dirnames, filenames in os.walk(split_dir):
        dirnames[:] = [name for name in dirnames if not name.startswith(".")]
        relative = os.path.relpath(dirpath, split_dir)
        if relative != "." and os.path.dirname(relative) == "":
            gloss_counts[split] += 1
        visible = [name for name in filenames if not name.startswith(".")]
        placed += len(visible)
        npy_names = [name for name in visible if name.endswith(".npy")]
        npy_counts[split] += len(npy_names)
        if not npy_names:
            continue
        sample = os.path.join(dirpath, npy_names[0])
        array = np.load(sample, mmap_mode="r")
        checked += 1
        if array.ndim != 3 or array.shape[1:] != (76, 3) or array.shape[0] == 0:
            bad_files.append((sample, tuple(int(dim) for dim in array.shape)))

for split in SPLITS:
    logger.info(
        "%s: %d glosses, %d npy files",
        split,
        gloss_counts[split],
        npy_counts[split],
    )

logger.info("Shape checks: %d", checked)
if bad_files:
    for path, shape in bad_files:
        logger.error("Unexpected array %s shape=%s", path, shape)
    raise RuntimeError(f"{len(bad_files)} files are not (T, 76, 3)")

if placed != len(jobs):
    raise RuntimeError(f"File count mismatch: planned {len(jobs)}, on disk {placed}")
logger.info("Output tree matches the plan: %d files under %s", placed, OUTPUT_ROOT)
